<a id="entropy-weight"></a>

$$
\text{熵权法 (Entropy Weight Method): 一种客观赋权法, 用数据本身的离散程度决定指标的权重} \\ \quad \\
\text{直觉: 如果所有评价对象在某个指标上的取值都差不多, 这个指标对区分好坏没有帮助, 权重应该小;} \\
\text{如果某个指标上各对象差别很大, 它提供的信息多, 权重应该大} \\
\text{"差不多" 和 "差别很大" 用信息熵 (Information Entropy) 来度量}
$$

<a id="information-entropy"></a>

$$
\text{1. 信息量 (Self-Information): 一个概率为 p 的事件发生时, 带来多少信息} \\
\text{要求: (1) 概率越小越意外, 信息量越大, I(p) 递减; (2) 必然事件 I(1) = 0;} \\
\text{(3) 两个独立事件同时发生, 信息量相加: } I(pq) = I(p) + I(q) \\
\text{满足 } f(xy) = f(x) + f(y) \text{ 的连续函数只有对数 (令 } g(t) = f(e^t) \text{, 则 } g(s + t) = g(s) + g(t) \text{, 只能是线性函数 } g(t) = ct \text{)} \\
I(p) = -\ln p \quad (\text{取负号保证递减, 底数 e 只影响单位}) \\ \quad \\
\text{2. 信息熵: 信息量的期望值, 衡量一个分布的 "不确定性"} \\
H = E[I] = \sum_{i=1}^n p_i I(p_i) = -\sum_{i=1}^n p_i \ln p_i \\
\text{约定 } 0 \ln 0 = 0 \text{ (洛必达法则: } \lim_{x \to 0^+} x \ln x = \lim_{x \to 0^+} \frac{\ln x}{1/x} = \lim_{x \to 0^+} \frac{1/x}{-1/x^2} = \lim_{x \to 0^+} (-x) = 0 \text{)} \\ \quad \\
\text{3. 熵的最大值: 拉格朗日乘子法} \\
\max H = -\sum p_i \ln p_i \quad \text{s.t.} \quad \sum p_i = 1 \\
L = -\sum p_i \ln p_i - \lambda \left(\sum p_i - 1\right) \\
\frac{\partial L}{\partial p_i} = -\ln p_i - 1 - \lambda = 0 \implies p_i = e^{-1-\lambda} \text{ (和 i 无关, 所有 } p_i \text{ 相等)} \\
\sum p_i = 1 \implies p_i = \frac{1}{n}, \quad H_{\max} = -\sum \frac{1}{n} \ln \frac{1}{n} = \ln n \\
\text{二阶条件: Hessian = diag}(-\frac{1}{p_1}, \cdots, -\frac{1}{p_n}) \text{ 负定, 所以是极大值} \\
\text{最小值 H = 0: 某个 } p_k = 1 \text{, 其余为 0 (完全确定, 没有不确定性)}
$$

数学基础: [拉格朗日乘子法](../math_basics/calculus.ipynb#lagrange) · [极值的二阶条件](../math_basics/calculus.ipynb#extremum) · [数学期望](../math_basics/statistics.ipynb#expectation)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import entropy as scipy_entropy

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

def information_entropy(p):
    p = np.asarray(p, dtype=float)
    nonzero = p > 0                       # 0 ln 0 = 0
    return -np.sum(p[nonzero] * np.log(p[nonzero])) + 0.0     # + 0.0 把 -0.0 变成 0.0

for p in [[0.25, 0.25, 0.25, 0.25], [0.7, 0.1, 0.1, 0.1], [0.97, 0.01, 0.01, 0.01], [1, 0, 0, 0]]:
    print(f'p = {p}: H = {information_entropy(p):.6f}, scipy = {scipy_entropy(p):.6f}')
print(f'ln 4 = {np.log(4):.6f} (最大值)')

# 随机生成很多分布, 熵都不超过 ln n
rng = np.random.default_rng(0)
random_p = rng.dirichlet(np.ones(4), size=100000)
H_random = -np.sum(random_p * np.log(random_p), axis=1)
print(f'10 万个随机分布的最大熵 = {H_random.max():.6f} <= ln 4')

# 两点分布的熵: p = 0.5 时最大
p_grid = np.linspace(0.001, 0.999, 400)
H_binary = -(p_grid * np.log(p_grid) + (1 - p_grid) * np.log(1 - p_grid))
plt.figure(figsize=(4, 3))
plt.plot(p_grid, H_binary, label='H(p, 1-p)')
plt.axhline(np.log(2), color='r', linestyle='--', label='ln 2')
plt.xlabel('p')
plot_nicely()
plt.show()

x_small = np.array([1e-1, 1e-3, 1e-6, 1e-9])
print(f'x ln x 当 x -> 0: {x_small * np.log(x_small)}')

<a id="entropy-weight-steps"></a>

$$
\text{熵权法的步骤 (n 个对象, m 个指标)} \\ \quad \\
\text{1. 正向化 (同 TOPSIS), 再标准化成非负数 } z_{ij} \geq 0 \text{ (因为后面要取对数, 不能有负数)} \\ \quad \\
\text{2. 把第 j 列看成一个概率分布: 第 i 个对象在该指标上的 "占比"} \\
p_{ij} = \frac{z_{ij}}{\sum_{i=1}^n z_{ij}}, \quad \sum_i p_{ij} = 1 \\ \quad \\
\text{3. 第 j 个指标的信息熵, 除以最大值 ln n 归一化到 [0, 1]:} \\
e_j = -\frac{1}{\ln n} \sum_{i=1}^n p_{ij} \ln p_{ij}, \quad 0 \leq e_j \leq 1 \\ \quad \\
\text{4. 信息效用值 (Degree of Divergence): } d_j = 1 - e_j \\
\text{所有对象取值相同 } \implies p_{ij} = \frac{1}{n} \implies e_j = 1 \implies d_j = 0 \text{ (完全无法区分对象, 不提供信息)} \\
\text{取值差别越大 } \implies \text{分布越不均匀} \implies e_j \text{ 越小} \implies d_j \text{ 越大} \\ \quad \\
\text{5. 权重: } w_j = \frac{d_j}{\sum_{k=1}^m d_k} \\ \quad \\
\text{注意: 熵权法只反映 "数据的离散程度", 不反映指标的实际重要性} \\
\text{(例如一个很重要但大家都做得很好的指标, 熵权会很小), 实际中常和 AHP 的主观权重组合使用}
$$

相关模型: [TOPSIS 的正向化和标准化](topsis.ipynb#forward-processing)

In [ ]:
def entropy_weight(Z):
    # Z: 正向化 + 标准化后的非负矩阵, n 行 (对象) m 列 (指标)
    n = Z.shape[0]
    P = Z / Z.sum(axis=0)
    with np.errstate(divide='ignore', invalid='ignore'):
        plnp = np.where(P > 0, P * np.log(P), 0.0)
    e = -plnp.sum(axis=0) / np.log(n)
    d = np.clip(1 - e, 0, None)          # 浮点误差可能让 e 略大于 1
    return d / d.sum(), e, d

# 例子: 河流水质 (和 TOPSIS 同一组数据)
rivers = ['A', 'B', 'C', 'D', 'E']
raw = np.array([[4.69, 6.59, 51, 11.94],
                [2.03, 7.86, 19, 6.46],
                [9.11, 6.31, 46, 8.91],
                [8.61, 7.05, 46, 26.43],
                [7.13, 6.50, 50, 23.57]])

def to_benefit_from_intermediate(x, best):
    return 1 - np.abs(x - best) / np.max(np.abs(x - best))

def to_benefit_from_interval(x, a, b):
    M = max(a - x.min(), x.max() - b)
    return 1 - np.where(x < a, a - x, np.where(x > b, x - b, 0.0)) / M

X_forward = np.column_stack([
    raw[:, 0],
    to_benefit_from_intermediate(raw[:, 1], 7),
    raw[:, 2].max() - raw[:, 2],
    to_benefit_from_interval(raw[:, 3], 10, 20),
])
Z = X_forward / np.linalg.norm(X_forward, axis=0)

w, e, d = entropy_weight(Z)
names = ['含氧量', 'PH', '细菌数', '营养物']
for name, ej, dj, wj in zip(names, e, d, w):
    print(f'{name}: 熵 e = {ej:.4f}, 效用 d = {dj:.4f}, 权重 w = {wj:.4f}')

# 独立验证: 每一列的熵用 scipy.stats.entropy (会自动归一化成概率) 再除以 ln n
e_scipy = np.array([scipy_entropy(Z[:, j]) for j in range(Z.shape[1])]) / np.log(Z.shape[0])
print(f'e == scipy ? {np.allclose(e, e_scipy)}')

# 极端情况: 常数列权重为 0
Z_const = np.column_stack([Z, np.full(5, 0.3)])
w_const, _, _ = entropy_weight(Z_const)
print(f'加入一个常数列后的权重 = {np.round(w_const, 4)} (最后一列为 0)')

<a id="entropy-topsis"></a>

$$
\text{熵权 TOPSIS: 用熵权法得到的 } w_j \text{ 作为 TOPSIS 距离中的权重} \\
D_i^+ = \sqrt{\sum_j w_j (Z_j^+ - z_{ij})^2}, \quad D_i^- = \sqrt{\sum_j w_j (Z_j^- - z_{ij})^2}, \quad C_i = \frac{D_i^-}{D_i^+ + D_i^-} \\ \quad \\
\text{组合赋权 (主观 + 客观): 例如乘法合成 } w_j = \frac{w_j^{AHP} \cdot w_j^{entropy}}{\sum_k w_k^{AHP} \cdot w_k^{entropy}} \\
\text{或线性组合 } w_j = \alpha w_j^{AHP} + (1 - \alpha) w_j^{entropy}
$$

In [ ]:
def topsis_score(Z, w):
    Z_plus, Z_minus = Z.max(axis=0), Z.min(axis=0)
    D_plus = np.sqrt(np.sum(w * (Z_plus - Z)**2, axis=1))
    D_minus = np.sqrt(np.sum(w * (Z_minus - Z)**2, axis=1))
    return D_minus / (D_plus + D_minus)

score_equal = topsis_score(Z, np.full(4, 0.25))
score_entropy = topsis_score(Z, w)
w_ahp = np.array([0.4, 0.1, 0.3, 0.2])
w_combined = w_ahp * w / np.sum(w_ahp * w)
score_combined = topsis_score(Z, w_combined)

print(f'组合权重 = {np.round(w_combined, 4)}')
for label, s in [('等权', score_equal), ('熵权', score_entropy), ('组合', score_combined)]:
    print(f'{label}: 得分 = {np.round(s, 4)}, 排名 = {[rivers[i] for i in np.argsort(-s)]}')

x_pos = np.arange(4)
plt.figure(figsize=(4, 3))
plt.bar(x_pos - 0.25, np.full(4, 0.25), width=0.25, label='equal')
plt.bar(x_pos, w, width=0.25, label='entropy')
plt.bar(x_pos + 0.25, w_combined, width=0.25, label='AHP x entropy')
plt.xticks(x_pos, ['O2', 'PH', 'bacteria', 'nutrient'])
plt.ylabel('weight')
plot_nicely()
plt.show()